# Reducción de Dimensionalidad aplicada a perfiles estudiantiles

## PCA vs t-SNE para análisis exploratorio y preparación de modelos predictivos

**Autor:** Héctor A. López Giménez  
**Área:** Data Science / Machine Learning / Educación  
**Tipo de proyecto:** Portafolio profesional

---

## Resumen ejecutivo

Este proyecto analiza registros académicos de estudiantes con el objetivo de reducir la dimensionalidad del dataset, visualizar patrones de comportamiento y evaluar si una representación más compacta puede apoyar futuros modelos predictivos.

Se comparan dos técnicas:

- **PCA (Principal Component Analysis):** técnica lineal útil para compresión, reducción de redundancia y preparación de modelos.
- **t-SNE (t-distributed Stochastic Neighbor Embedding):** técnica no lineal orientada a visualización exploratoria de estructuras locales.

El análisis demuestra que PCA conserva gran parte de la información con pocos componentes, mientras que t-SNE permite observar agrupamientos visuales más definidos. Para un flujo predictivo reproducible, PCA resulta más adecuado; para exploración y comunicación visual, t-SNE aporta mayor claridad.

## 1. Planteamiento del problema

Una organización educativa cuenta con registros de estudiantes que incluyen variables académicas y demográficas. El objetivo es explorar si es posible representar esos datos en menos dimensiones para:

- visualizar perfiles estudiantiles;
- detectar patrones ocultos;
- reducir redundancia entre variables;
- preparar los datos para modelos predictivos;
- comunicar resultados de forma clara a perfiles técnicos y no técnicos.

Este proyecto trabaja sobre un dataset con variables como edad, horas de estudio, evaluaciones aprobadas, participación en foros y tareas entregadas.

## 2. Importación de librerías

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## 3. Carga del dataset

In [2]:
# El notebook está preparado para ejecutarse en GitHub, Colab o Jupyter.
# Coloca el archivo prueba_estudiantes.csv en la raíz del repositorio o en data/.

candidate_paths = [
    "data/prueba_estudiantes.csv",
    "prueba_estudiantes.csv",
    "../data/prueba_estudiantes.csv",
    "/mnt/data/prueba_estudiantes.csv"
]

DATA_PATH = None
for path in candidate_paths:
    if os.path.exists(path):
        DATA_PATH = path
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "No se encontró prueba_estudiantes.csv. Ubícalo en la raíz del repositorio o en data/."
    )

df = pd.read_csv(DATA_PATH)

print(f"Archivo cargado desde: {DATA_PATH}")
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head()

Archivo cargado desde: data/prueba_estudiantes.csv
Filas: 500 | Columnas: 6


,Edad,Horas_estudio,Evaluaciones_aprobadas,Participacion_foro,Tareas_entregadas,Perfil
0,36,6.500,5,3,3,en_riesgo
1,22,7.800,7,5,6,rendimiento_medio
2,36,7.800,8,7,7,rendimiento_medio
3,43,3.700,3,0,3,bajo_rendimiento
4,26,5.200,4,5,8,en_riesgo


## 4. Exploración inicial y calidad de datos

In [3]:
print("Columnas del dataset:")
print(df.columns.tolist())

print("\nTipos de datos:")
display(df.dtypes.to_frame("tipo_dato"))

print("\nValores faltantes por columna:")
display(df.isnull().sum().to_frame("faltantes"))

print(f"\nDuplicados: {df.duplicated().sum()}")

print("\nEstadísticas descriptivas:")
display(df.describe().T)

Columnas del dataset:
['Edad', 'Horas_estudio', 'Evaluaciones_aprobadas', 'Participacion_foro', 'Tareas_entregadas', 'Perfil']

Tipos de datos:


,tipo_dato
Edad,int64
Horas_estudio,float64
Evaluaciones_aprobadas,int64
Participacion_foro,int64
Tareas_entregadas,int64
Perfil,object



Valores faltantes por columna:


,faltantes
Edad,0
Horas_estudio,0
Evaluaciones_aprobadas,0
Participacion_foro,0
Tareas_entregadas,0
Perfil,0



Duplicados: 0

Estadísticas descriptivas:


,count,mean,std,min,25%,50%,75%,max
Edad,500.000,31.916,9.208,17.000,25.000,31.000,39.000,54.000
Horas_estudio,500.000,7.569,4.337,0.500,4.000,7.150,11.700,18.500
Evaluaciones_aprobadas,500.000,5.590,2.819,0.000,3.000,5.000,8.000,12.000
Participacion_foro,500.000,5.594,2.955,0.000,3.000,6.000,8.000,12.000
Tareas_entregadas,500.000,6.828,3.667,0.000,4.000,7.000,10.000,12.000


### Observaciones iniciales

El dataset contiene variables numéricas relacionadas con desempeño y participación estudiantil, además de una variable categórica `Perfil`, que se utilizará como referencia visual y para la validación predictiva.

La reducción de dimensionalidad se aplicará únicamente sobre las variables numéricas.

In [4]:
features = [
    "Edad",
    "Horas_estudio",
    "Evaluaciones_aprobadas",
    "Participacion_foro",
    "Tareas_entregadas"
]

target = "Perfil"

X = df[features].copy()
y = df[target].copy()

print("Variables numéricas utilizadas:")
print(features)

print("\nDistribución de perfiles:")
display(y.value_counts().to_frame("cantidad"))

Variables numéricas utilizadas:
['Edad', 'Horas_estudio', 'Evaluaciones_aprobadas', 'Participacion_foro', 'Tareas_entregadas']

Distribución de perfiles:


,cantidad
Perfil,
alto_rendimiento,153
rendimiento_medio,132
bajo_rendimiento,114
en_riesgo,101


## 5. Análisis exploratorio de variables

In [5]:
for col in features:
    plt.figure(figsize=(7, 4))
    plt.hist(df[col], bins=25, edgecolor="black", alpha=0.8)
    plt.title(f"Distribución de {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

<Figure size 700x400 with 1 Axes>

<Figure size 700x400 with 1 Axes>

<Figure size 700x400 with 1 Axes>

<Figure size 700x400 with 1 Axes>

<Figure size 700x400 with 1 Axes>

In [6]:
corr = df[features].corr()

plt.figure(figsize=(7, 5))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlación")
plt.xticks(range(len(features)), features, rotation=45, ha="right")
plt.yticks(range(len(features)), features)

for i in range(len(features)):
    for j in range(len(features)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")

plt.title("Matriz de correlación entre variables numéricas")
plt.tight_layout()
plt.show()

display(corr)

<Figure size 700x500 with 2 Axes>

,Edad,Horas_estudio,Evaluaciones_aprobadas,Participacion_foro,Tareas_entregadas
Edad,1.000,-0.818,-0.825,-0.743,-0.832
Horas_estudio,-0.818,1.000,0.868,0.798,0.858
Evaluaciones_aprobadas,-0.825,0.868,1.000,0.785,0.843
Participacion_foro,-0.743,0.798,0.785,1.000,0.785
Tareas_entregadas,-0.832,0.858,0.843,0.785,1.000


### Lectura del EDA

Las variables de comportamiento académico presentan una relación importante entre sí. Esto justifica el uso de PCA, ya que esta técnica puede transformar variables correlacionadas en componentes principales no correlacionados, reduciendo redundancia sin descartar directamente columnas originales.

## 6. Escalamiento de variables

In [7]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=features)

display(X_scaled_df.describe().round(3))

,Edad,Horas_estudio,Evaluaciones_aprobadas,Participacion_foro,Tareas_entregadas
count,500.000,500.000,500.000,500.000,500.000
mean,-0.000,-0.000,0.000,-0.000,-0.000
std,1.001,1.001,1.001,1.001,1.001
min,-1.622,-1.632,-1.985,-1.895,-1.864
25%,-0.752,-0.824,-0.920,-0.879,-0.772
50%,-0.100,-0.097,-0.210,0.138,0.047
75%,0.770,0.954,0.856,0.815,0.866
max,2.401,2.523,2.276,2.170,1.412


El escalamiento es necesario porque PCA y t-SNE son sensibles a la escala de las variables. Si una variable tiene magnitudes mayores, puede dominar el cálculo de varianza o distancia.

## 7. PCA: reducción lineal de dimensionalidad

In [8]:
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X_scaled)

print("Varianza explicada por PCA 2D:")
for i, ratio in enumerate(pca_2d.explained_variance_ratio_, start=1):
    print(f"PC{i}: {ratio * 100:.2f}%")

print(f"\nVarianza explicada acumulada PCA 2D: {pca_2d.explained_variance_ratio_.sum() * 100:.2f}%")
print(f"Varianza explicada acumulada PCA 3D: {pca_3d.explained_variance_ratio_.sum() * 100:.2f}%")

Varianza explicada por PCA 2D:
PC1: 85.27%
PC2: 5.35%

Varianza explicada acumulada PCA 2D: 90.62%
Varianza explicada acumulada PCA 3D: 94.33%


In [9]:
pca_full = PCA()
pca_full.fit(X_scaled)

varianza = pca_full.explained_variance_ratio_
varianza_acumulada = np.cumsum(varianza)

tabla_varianza = pd.DataFrame({
    "Componente": [f"PC{i}" for i in range(1, len(varianza) + 1)],
    "Varianza_explicada": varianza,
    "Varianza_acumulada": varianza_acumulada
})

display(tabla_varianza)

,Componente,Varianza_explicada,Varianza_acumulada
0,PC1,0.853,0.853
1,PC2,0.053,0.906
2,PC3,0.037,0.943
3,PC4,0.031,0.974
4,PC5,0.026,1.000


In [10]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(varianza_acumulada) + 1), varianza_acumulada, marker="o")
plt.axhline(0.90, linestyle="--", label="90% varianza")
plt.axhline(0.95, linestyle="--", label="95% varianza")
plt.title("Varianza explicada acumulada - PCA")
plt.xlabel("Número de componentes")
plt.ylabel("Varianza acumulada")
plt.xticks(range(1, len(varianza_acumulada) + 1))
plt.ylim(0, 1.05)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

<Figure size 700x400 with 1 Axes>

## 8. Visualización PCA en 2D

In [11]:
plt.figure(figsize=(8, 6))

for perfil in df[target].unique():
    mask = df[target] == perfil
    plt.scatter(
        X_pca_2d[mask, 0],
        X_pca_2d[mask, 1],
        s=35,
        alpha=0.7,
        label=perfil
    )

plt.title("Visualización PCA 2D por perfil estudiantil")
plt.xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0] * 100:.2f}% varianza)")
plt.ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1] * 100:.2f}% varianza)")
plt.grid(alpha=0.25)
plt.legend(title="Perfil")
plt.tight_layout()
plt.show()

<Figure size 800x600 with 1 Axes>

## 9. Interpretación de componentes principales

In [12]:
loadings = pd.DataFrame(
    pca_2d.components_.T,
    columns=["PC1", "PC2"],
    index=features
)

display(loadings)

plt.figure(figsize=(7, 5))
plt.imshow(loadings, aspect="auto")
plt.colorbar(label="Peso del loading")
plt.xticks(range(loadings.shape[1]), loadings.columns)
plt.yticks(range(loadings.shape[0]), loadings.index)

for i in range(loadings.shape[0]):
    for j in range(loadings.shape[1]):
        plt.text(j, i, f"{loadings.iloc[i, j]:.2f}", ha="center", va="center")

plt.title("Loadings PCA: peso de cada variable")
plt.tight_layout()
plt.show()

,PC1,PC2
Edad,-0.442,0.477
Horas_estudio,0.456,-0.045
Evaluaciones_aprobadas,0.454,-0.137
Participacion_foro,0.430,0.852
Tareas_entregadas,0.453,-0.162


<Figure size 700x500 with 2 Axes>

### Interpretación PCA

Los dos primeros componentes explican aproximadamente el **90% de la variabilidad total**, por lo que la visualización 2D conserva gran parte de la información del dataset.

- **PC1** concentra principalmente variables académicas como horas de estudio, evaluaciones aprobadas, participación en foro y tareas entregadas. Puede interpretarse como un eje general de **compromiso y desempeño académico**.
- **PC2** captura diferencias más específicas asociadas a participación y edad, ayudando a separar algunos perfiles que se solapan en PC1.

Desde una perspectiva de negocio educativo, PCA permite identificar grupos de estudiantes con distintos niveles de avance y participación. Esto puede apoyar acciones como tutorías tempranas, rutas de aprendizaje personalizadas o seguimiento preventivo.

## 10. t-SNE: visualización no lineal

t-SNE es una técnica no lineal de reducción dimensional orientada principalmente a visualización. A diferencia de PCA, no se centra en maximizar varianza explicada, sino en preservar relaciones locales de vecindad.

Como t-SNE es sensible al parámetro `perplexity`, se prueban varios valores para evaluar la estabilidad visual de los perfiles.

In [13]:
perfil_codes = df[target].astype("category").cat.codes.values

perplexities = [5, 15, 30, 50]
tsne_results = {}
tsne_silhouettes = {}

for perplexity in perplexities:
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        learning_rate="auto",
        init="pca",
        random_state=42
    )

    X_tsne = tsne.fit_transform(X_scaled)
    tsne_results[perplexity] = X_tsne
    tsne_silhouettes[perplexity] = silhouette_score(X_tsne, perfil_codes)

pd.DataFrame({
    "perplexity": list(tsne_silhouettes.keys()),
    "silhouette_visual_referencia": list(tsne_silhouettes.values())
})

,perplexity,silhouette_visual_referencia
0,5,0.578
1,15,0.641
2,30,0.662
3,50,0.683


In [14]:
for perplexity in perplexities:
    Xt = tsne_results[perplexity]

    plt.figure(figsize=(7, 5))
    for perfil in df[target].unique():
        mask = df[target] == perfil
        plt.scatter(
            Xt[mask, 0],
            Xt[mask, 1],
            s=25,
            alpha=0.7,
            label=perfil
        )

    plt.title(
        f"t-SNE 2D por perfil | perplexity={perplexity} | "
        f"Silhouette ref.={tsne_silhouettes[perplexity]:.3f}"
    )
    plt.xlabel("Dimensión t-SNE 1")
    plt.ylabel("Dimensión t-SNE 2")
    plt.grid(alpha=0.25)
    plt.legend(title="Perfil")
    plt.tight_layout()
    plt.show()

<Figure size 700x500 with 1 Axes>

<Figure size 700x500 with 1 Axes>

<Figure size 700x500 with 1 Axes>

<Figure size 700x500 with 1 Axes>

In [15]:
best_perplexity = max(tsne_silhouettes, key=tsne_silhouettes.get)
X_tsne_best = tsne_results[best_perplexity]

print(f"Perplexity con mayor separación visual usando Perfil como referencia: {best_perplexity}")
print(f"Silhouette de referencia: {tsne_silhouettes[best_perplexity]:.3f}")

Perplexity con mayor separación visual usando Perfil como referencia: 50
Silhouette de referencia: 0.683


## 11. Comparación visual PCA vs t-SNE

In [16]:
plt.figure(figsize=(8, 6))
for perfil in df[target].unique():
    mask = df[target] == perfil
    plt.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], s=35, alpha=0.7, label=perfil)

plt.title("PCA 2D - estructura global")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(alpha=0.25)
plt.legend(title="Perfil")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
for perfil in df[target].unique():
    mask = df[target] == perfil
    plt.scatter(X_tsne_best[mask, 0], X_tsne_best[mask, 1], s=35, alpha=0.7, label=perfil)

plt.title(f"t-SNE 2D - estructura local | perplexity={best_perplexity}")
plt.xlabel("Dimensión t-SNE 1")
plt.ylabel("Dimensión t-SNE 2")
plt.grid(alpha=0.25)
plt.legend(title="Perfil")
plt.tight_layout()
plt.show()

<Figure size 800x600 with 1 Axes>

<Figure size 800x600 with 1 Axes>

### Comparación interpretativa

**PCA** muestra una separación ordenada de los perfiles a lo largo de un eje general de desempeño. Es más interpretable porque sus componentes pueden analizarse mediante loadings y varianza explicada.

**t-SNE** genera grupos visualmente más compactos y separados, especialmente útil para comunicar la presencia de agrupamientos locales. Sin embargo, sus ejes no tienen interpretación directa y el resultado depende del ajuste de hiperparámetros como `perplexity`.

En síntesis:

- PCA es preferible para **preprocesamiento y modelado predictivo**.
- t-SNE es preferible para **exploración visual y comunicación de patrones**.

## 12. Tabla comparativa PCA vs t-SNE

| Criterio | PCA | t-SNE |
|---|---|---|
| Tipo de técnica | Lineal | No lineal |
| Objetivo principal | Reducción, compresión y preprocesamiento | Visualización exploratoria |
| Qué preserva | Varianza y estructura global | Vecindades y estructura local |
| Interpretabilidad | Alta mediante loadings | Baja; los ejes no tienen significado directo |
| Varianza explicada | Sí | No |
| Uso en modelos predictivos | Recomendado | No recomendado como entrada estable |
| Transformación de nuevos datos | Sí, mediante `transform()` | No de forma convencional |
| Sensibilidad a parámetros | Baja-moderada | Alta (`perplexity`, `learning_rate`) |
| Ventaja principal | Reduce redundancia y multicolinealidad | Revela agrupamientos locales complejos |
| Desventaja principal | Puede perder relaciones no lineales | Resultado sensible e interpretación limitada |
| Caso ideal | Pipelines de ML, regresión, clasificación, clustering | Exploración visual, storytelling, detección visual de grupos |
| Ejemplo en este proyecto | Compactar variables académicas en componentes | Visualizar perfiles estudiantiles en 2D |

## 13. Validación predictiva con y sin PCA

Para evaluar la utilidad de PCA en un contexto predictivo, se compara un modelo de clasificación usando las variables originales contra el mismo modelo usando PCA con 3 componentes.

Esta comparación no busca crear el mejor modelo posible, sino verificar si la reducción dimensional mantiene información suficiente para clasificar perfiles estudiantiles.

In [17]:
y_codes = df[target].astype("category").cat.codes.values

model_base = RandomForestClassifier(random_state=42)

scores_base = cross_val_score(
    model_base,
    X.values,
    y_codes,
    cv=5,
    scoring="accuracy"
)

pipeline_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=3)),
    ("model", RandomForestClassifier(random_state=42))
])

scores_pca = cross_val_score(
    pipeline_pca,
    X.values,
    y_codes,
    cv=5,
    scoring="accuracy"
)

resultados_modelo = pd.DataFrame({
    "Modelo": ["Random Forest sin PCA", "Random Forest con PCA 3D"],
    "Accuracy_promedio": [scores_base.mean(), scores_pca.mean()],
    "Accuracy_std": [scores_base.std(), scores_pca.std()]
})

display(resultados_modelo)

,Modelo,Accuracy_promedio,Accuracy_std
0,Random Forest sin PCA,0.966,0.021
1,Random Forest con PCA 3D,0.974,0.010


In [18]:
plt.figure(figsize=(7, 4))
plt.bar(resultados_modelo["Modelo"], resultados_modelo["Accuracy_promedio"])
plt.ylabel("Accuracy promedio")
plt.title("Comparación predictiva con y sin PCA")
plt.ylim(0, 1.05)
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

<Figure size 700x400 with 1 Axes>

### Justificación técnica final

Para preparar un modelo predictivo, la técnica más adecuada en este caso es **PCA**.

La razón principal es que PCA produce una transformación reproducible, interpretable y aplicable a nuevos datos. Además, permite reducir las variables originales a componentes principales que conservan la mayor parte de la información. En este dataset, tres componentes conservan cerca del 94% de la varianza, por lo que se logra una representación más compacta con pérdida limitada.

t-SNE es muy útil para visualizar agrupamientos, pero no es la opción adecuada como etapa de preprocesamiento para producción predictiva. Sus ejes no tienen interpretación directa, es sensible a hiperparámetros y no está pensado para transformar nuevos registros de forma convencional.

El experimento predictivo muestra que el desempeño con PCA se mantiene competitivo frente al uso de variables originales, lo que refuerza su utilidad como paso previo en pipelines de Machine Learning.

## 14. Conclusiones del proyecto

1. El dataset presenta variables académicas correlacionadas, lo que justifica aplicar reducción de dimensionalidad.
2. PCA logró representar la mayor parte de la información con pocos componentes.
3. Los loadings permiten interpretar PC1 como un eje general de desempeño y compromiso académico.
4. t-SNE mostró agrupamientos locales más visibles y útiles para comunicación exploratoria.
5. Para modelos predictivos, PCA es más apropiado porque es reproducible, interpretable y puede integrarse en pipelines.
6. Para storytelling, análisis visual y detección de patrones locales, t-SNE es una herramienta complementaria muy valiosa.

---

## Próximos pasos sugeridos

- Probar otros modelos predictivos como Logistic Regression, SVM o KNN.
- Comparar PCA con selección de variables.
- Evaluar pipelines con validación cruzada estratificada.
- Crear un dashboard con visualización de perfiles estudiantiles.
- Desplegar el análisis como caso de portafolio en GitHub con README y dataset.

## Tecnologías utilizadas

- Python
- Pandas
- NumPy
- Matplotlib
- Scikit-learn
- PCA
- t-SNE
- Random Forest
- Validación cruzada